In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

# Lecture Bronze
df = spark.table("pharma_catalog.bronze.bronze_supplier_orders")
display(df)

In [0]:
# COMMAND ----------
# Déduplication
df_dedup = df.dropDuplicates(["supplier_order_id"])
print(f"Avant: {df.count()} | Après dédup: {df_dedup.count()}")

# Contrôles qualité
df_quality = df_dedup.withColumn("rule_failed",
    F.when(F.col("supplier_order_id").isNull(), F.lit("null_supplier_order_id"))
     .when(F.col("supplier_id").isNull(), F.lit("null_supplier_id"))
     .when(F.col("product_id").isNull(), F.lit("null_product_id"))
     .when(F.col("warehouse_id").isNull(), F.lit("null_warehouse_id"))
     .when(F.col("order_date").isNull(), F.lit("null_order_date"))
     .when(F.col("quantity_ordered") <= 0, F.lit("quantite_invalide"))
     .when(F.col("quantity_received") < 0, F.lit("reception_negative"))
     .when(F.col("quantity_received") > F.col("quantity_ordered"), F.lit("reception_superieure_commande"))
     .when(
         F.col("expected_delivery_date").isNotNull() & 
         (F.col("expected_delivery_date") < F.col("order_date")), 
         F.lit("date_livraison_anterieure_commande")
     )
     .when(F.col("order_status").isNull(), F.lit("null_order_status"))
     .otherwise(F.lit(None))
)

passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f" Lignes OK : {passed.count()}")
print(f" Lignes rejetées : {failed.count()}")
display(failed.select("supplier_order_id", "quantity_ordered", "quantity_received", "order_date", "expected_delivery_date", "rule_failed"))

In [0]:
# COMMAND ----------
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_supplier_orders")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))
    df_quarantine.write.mode("append") \
                       .option("mergeSchema", "true") \
                       .saveAsTable("pharma_catalog.silver_quarantine.quarantine")
else:
    print("✅ Aucune ligne rejetée — quarantine vide")



In [0]:
# COMMAND ----------
passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_supplier_orders")
print(f" {passed.count()} ligne(s) chargée(s) dans silver_supplier_orders")

In [0]:
# COMMAND ----------
# Bloc 5 — Vérification finale
display(spark.table("pharma_catalog.silver.silver_supplier_orders"))

# COMMAND ----------
display(spark.table("pharma_catalog.silver_quarantine.quarantine"))